# CS 3110/5110: Data Privacy
## Homework 5

In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def laplace_mech_vec(qs, sensitivity, epsilon):
    return [laplace_mech(q, sensitivity, epsilon) for q in qs]

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(qs, sensitivity, epsilon, delta):
    return [gaussian_mech(q, sensitivity, epsilon, delta) for q in qs]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Question 1 (5 points)

Implement the `dp_occupation_histogram` function below. It should return a differentially private histogram over the `Occupation` column in the `adult` dataset. Your function should have a total privacy cost of `epsilon` and should use parallel composition.

In [2]:
def dp_occupation_histogram(epsilon):
    hist = adult.groupby(["Occupation"]).size()
    return hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))

dp_occupation_histogram(1.0)

Occupation
Adm-clerical         3769.182741
Armed-Forces            8.885109
Craft-repair         4099.184362
Exec-managerial      4065.629819
Farming-fishing       993.046907
Handlers-cleaners    1369.983888
Machine-op-inspct    2002.546956
Other-service        3294.038604
Priv-house-serv       147.964406
Prof-specialty       4141.539615
Protective-serv       648.991393
Sales                3649.348375
Tech-support          928.578553
Transport-moving     1598.337586
dtype: float64

In [3]:
# TEST CASE
hist = adult['Occupation'].value_counts()
dp_results = [np.abs(dp_occupation_histogram(1.0)-hist) for _ in range(200)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1/1.0) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 2
assert stats.wasserstein_distance(all_dp_results, spec) > 0

dp_results = [np.abs(dp_occupation_histogram(0.01)-hist) for _ in range(200)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1/0.01) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 150
assert stats.wasserstein_distance(all_dp_results, spec) > 50

## Question 2 (10 points)

In 2-5 sentences each, answer the following:

- Argue that your solution for `dp_occupation_histogram` satisfies `epsilon`-differential privacy.

- How would your results change (in terms of accuracy) if you used sequential composition instead of parallel composition to implement `dp_occupation_histogram`, and why?

- If you partition the data by *both* occupation and age (i.e. a contingency table), would parallel composition still apply? Why or why not?

- If you partition the data by both occupation and age, what would happen to accuracy, and why?


By the rules of parallel composition, dp_occupation_histogram satisfies epsilon. The adult dataset is split into disjoint subsets and the Laplace mech is applied with epsilon to each subset. The sensitivity is 1 because adding or removing 1 person from the dataset would change the count of 1 subset by 1 while not affecting the other subsets. 

If sequential composition was used instead, the accuracy would be lower. It would be lower because the epsilon would have to be split into smaller parts when used to add noise, so the noise added would be greater.

If the data was partitioned by both occupation and age parallel compositon would still apply. This is the case because the data would be split into a disjoint subsets by occupation, then into smaller disjoint subsets by age.

If the data was partitioned on data by both occupation and age the accuracy would be lower. This is because each partiton would have sparser data, so the noise added would be relatively higher. 


## Question 3 (5 points)

Implement the `dp_3d_histogram` function below. It should return a differentially private 3-dimensional histogram over the `Occupation`, `Education`, and `Sex` columns (in that order) in the `adult` dataset (i.e. one count for each combination of occupation, education, and sex). As in Question 1, your function should have a total privacy cost of `epsilon` and should use parallel composition.

*Hint*: `value_counts` works on multiple columns.

In [21]:
def dp_3d_histogram(epsilon):
    # hist = adult.groupby(["Occupation", "Education", "Sex"]).size()
    hist = adult.value_counts(["Occupation", "Education", "Sex"])
    return hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))

dp_3d_histogram(1.0)

Occupation         Education     Sex   
Craft-repair       HS-grad       Male      1820.978884
Exec-managerial    Bachelors     Male      1038.101061
Adm-clerical       HS-grad       Female     968.501522
Prof-specialty     Bachelors     Male       915.457628
Adm-clerical       Some-college  Female     905.077731
                                              ...     
Transport-moving   Doctorate     Male         1.238456
Protective-serv    9th           Female       0.230454
Other-service      Prof-school   Female       1.579131
Handlers-cleaners  1st-4th       Female       0.090347
Machine-op-inspct  Doctorate     Female       0.004759
Name: count, Length: 361, dtype: float64

In [22]:
# TEST CASE
hist = adult[['Occupation', 'Education', 'Sex']].value_counts()
dp_results = [(dp_3d_histogram(1.0)-hist).dropna() for _ in range(50)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1/1.0) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 0.3
assert stats.wasserstein_distance(all_dp_results, spec) > 0

dp_results = [(dp_3d_histogram(0.1)-hist).dropna() for _ in range(50)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1/0.1) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 3
assert stats.wasserstein_distance(all_dp_results, spec) > 0

## Question 4 (10 points)

In 2-5 sentences each, answer the following:

- Argue that your solution for `dp_3d_histogram` satisfies `epsilon`-differential privacy.
- Which histogram do you expect to have higher relative error (percent error): the 1-dimensional one from Question 1, or the 3-dimensional one from Question 3? Why?


By the rules of parallel composition, dp_occupation_histogram satisfies epsilon. The adult dataset is split into disjoint subsets and again, into more disjoint subsets. The Laplace mech is applied with epsilon to each subset. The sensitivity is 1 because adding or removing 1 person from the dataset would change the count of 1 subset by 1 while not affecting the other subsets. 



The 3-dimensional one has higher error.

## Question 5 (5 points)

Implement the `dp_three_histograms` function below. It should return a tuple of three differentially private histograms:

1. A 1-dimensional histogram over the `Occupation` column
2. A 2-dimensional histogram over the `Occupation` and `Education` columns
3. A 3-dimensional histogram over the `Occupation`, `Education`, and `Sex` columns

Your function should have a total privacy cost of `epsilon`.

Hint: take care with composition!

In [28]:
def dp_three_histograms(epsilon):
    hists = []
    hist_cols = (["Occupation"], ["Occupation", "Education"], ["Occupation", "Education", "Sex"])

    for hc in hist_cols:
        hist = adult.groupby(hc).size()
        # hist = adult.value_counts(hc)
        hists.append(hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon/3)))

    return tuple(hists)

dp_three_histograms(1.0)

(Occupation
 Adm-clerical         3768.050299
 Armed-Forces            8.992079
 Craft-repair         4103.318682
 Exec-managerial      4065.430151
 Farming-fishing       994.581799
 Handlers-cleaners    1375.299634
 Machine-op-inspct    1998.918315
 Other-service        3295.507793
 Priv-house-serv       149.449593
 Prof-specialty       4139.958614
 Protective-serv       653.022440
 Sales                3647.637961
 Tech-support          928.996125
 Transport-moving     1603.329091
 dtype: float64,
 Occupation        Education   
 Adm-clerical      10th             34.869745
                   11th             67.508167
                   12th             35.540296
                   5th-6th          11.678431
                   7th-8th          12.459439
                                      ...    
 Transport-moving  Doctorate        -7.323942
                   HS-grad         827.596928
                   Masters           9.241010
                   Prof-school       1.129682
   

In [29]:
# TEST CASE
hists = [adult['Occupation'].value_counts(),
         adult[['Occupation', 'Education']].value_counts(),
         adult[['Occupation', 'Education', 'Sex']].value_counts()]
dp_results = [dp_three_histograms(1.0) for _ in range(100)]
assert all(len(r) == 3 for r in dp_results)

for i, hist in enumerate(hists):
    all_dp_results = [item for r in dp_results for item in (r[i]-hist).dropna()]
    spec = [np.random.laplace(loc=0, scale=3/1.0) for _ in range(2000)]
    assert stats.wasserstein_distance(all_dp_results, spec) < 1
    assert stats.wasserstein_distance(all_dp_results, spec) > 0

## Question 6 (5 points)

Using the 3-point structure described in lecture, argue that your solution for `dp_three_histograms` satisfies `epsilon`-differential privacy. Be precise about composition.



Sensitivty: The sens? 1 or 2, adding or removing from one to another?
Epsilon: 
Post Processing: No way to undo DP 